# Create Dashboard using Plotly and Dash

## Objectives
After completing this lab, you will be able to:

- Create a dashboard using Dash and Plotly
- Add dropdown menus for interactive dashboard filtering
- Create callbacks for dynamic visualization updates
- Display multiple charts in a dashboard layout
- Analyze automobile sales during recession and non-recession periods

---

## Scenario

You are working as a data scientist for **XYZAutomotives**.  
The company wants to analyze automobile sales trends during recession periods and compare them with yearly sales statistics.

You will create an interactive dashboard with:

1. **Yearly Automobile Sales Statistics**
2. **Recession Period Statistics**

The dashboard will contain multiple interactive visualizations using Plotly and Dash.

In [3]:
# Install required packages (Run only once if needed)

# !pip install dash
# !pip install plotly
# !pip install pandas
# !pip install more-itertools

# Import Required Libraries

We will import:

- pandas
- dash
- dash html and dcc components
- dash callback dependencies
- plotly.express

In [4]:
# Import required libraries

import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Load Dataset

We will use the Automobile Sales dataset provided by IBM Skills Network.

In [5]:
import requests
import io
import pandas as pd

# URL of the CSV file
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/d51iMGfp_t0QpO30Lym-dw/automobile-sales.csv"

# Fetch the data from the URL
response = requests.get(URL)

# Raise an error if the request failed
response.raise_for_status()

# Convert the response content into a readable format for pandas
csv_content = io.StringIO(response.text)

# Read the CSV data into a pandas dataframe
data = pd.read_csv(csv_content)

# Print confirmation
print('Data downloaded and read into a dataframe!')

# Optional: Show the first few rows of the dataframe
print(data.head())

Data downloaded and read into a dataframe!
         Date  Year Month  Recession  Consumer_Confidence  Seasonality_Weight  \
0  1980-01-31  1980   Jan          1               108.24                0.45   
1  1980-01-31  1980   Jan          1               108.24                0.45   
2  1980-01-31  1980   Jan          1               108.24                0.36   
3  1980-01-31  1980   Jan          1               108.24                0.38   
4  1980-02-29  1980   Feb          1                98.75                0.46   

   Price  Advertising_Expenditure  Competition    GDP  Growth_Rate  \
0  27704                   1417.5            7  60.22         0.01   
1  77270                    763.7            7  60.22         0.01   
2  19665                   1417.5            7  60.22         0.01   
3  36986                   1417.5            7  60.22         0.01   
4  26609                   2773.4            4  45.99        -0.31   

   unemployment_rate  Automobile_Sales     Vehicl

# Check Dataset Information

In [6]:
# Dataset shape

print("Dataset Shape:", data.shape)

# Dataset columns
print("\nColumns:\n")
print(data.columns)

Dataset Shape: (2112, 15)

Columns:

Index(['Date', 'Year', 'Month', 'Recession', 'Consumer_Confidence',
       'Seasonality_Weight', 'Price', 'Advertising_Expenditure', 'Competition',
       'GDP', 'Growth_Rate', 'unemployment_rate', 'Automobile_Sales',
       'Vehicle_Type', 'City'],
      dtype='object')


# Create Year List for Dropdown

In [7]:
# Create list of years

year_list = [i for i in range(1980, 2014, 1)]

year_list[:5]

[1980, 1981, 1982, 1983, 1984]

# Create Dash Application

In [8]:
# Create Dash app
from dash import Dash
app = Dash(__name__)

# Suppress callback exceptions
app.config.suppress_callback_exceptions = True

# Dashboard Layout

The dashboard contains:

- Main title
- Report type dropdown
- Year dropdown
- Output container for charts

In [9]:
# App Layout

app.layout = html.Div([

    # Dashboard Title
    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={
            'textAlign': 'center',
            'color': '#503D36',
            'font-size': 24
        }
    ),

    # Dropdown Section
    html.Div([

        # Report Type Dropdown
        html.Div([
            html.Label("Select Statistics:"),
            dcc.Dropdown(
                id='dropdown-statistics',

                options=[
                    {
                        'label': 'Yearly Statistics',
                        'value': 'Yearly Statistics'
                    },

                    {
                        'label': 'Recession Period Statistics',
                        'value': 'Recession Period Statistics'
                    }
                ],

                placeholder='Select a report type',
                value='Select Statistics',

                style={
                    'width': '80%',
                    'padding': '3px',
                    'font-size': '20px',
                    'text-align-last': 'center'
                }
            )
        ]),

        html.Br(),

        # Year Dropdown
        html.Div([
            html.Label("Select Year:"),

            dcc.Dropdown(
                id='select-year',

                options=[
                    {'label': i, 'value': i}
                    for i in year_list
                ],

                placeholder='Select-year',
                value='Select-year'
            )
        ])

    ]),

    html.Br(),

    # Output Container
    html.Div([
        html.Div(
            id='output-container',
            className='chart-grid',
            style={'display': 'flex', 'flex-direction': 'column'}
        )
    ])

])

# Callback 1 – Enable or Disable Year Dropdown

This callback controls whether the year dropdown should be enabled.

- Enabled for Yearly Statistics
- Disabled for Recession Statistics

In [10]:
# Callback to enable/disable year dropdown

@app.callback(

    Output(
        component_id='select-year',
        component_property='disabled'
    ),

    Input(
        component_id='dropdown-statistics',
        component_property='value'
    )
)

def update_input_container(selected_statistics):

    if selected_statistics == 'Yearly Statistics':
        return False

    else:
        return True

# Callback 2 – Update Dashboard Charts

This callback dynamically generates charts based on:

- Selected report type
- Selected year

In [13]:
# Callback for output container

@app.callback(

    Output(
        component_id='output-container',
        component_property='children'
    ),

    [
        Input(
            component_id='dropdown-statistics',
            component_property='value'
        ),

        Input(
            component_id='select-year',
            component_property='value'
        )
    ]
)

def update_output_container(selected_statistics, input_year):

    # Recession Report
    if selected_statistics == 'Recession Period Statistics':

        recession_data = data[data['Recession'] == 1]

        # ------------------------------
        # Plot 1
        # Average Automobile Sales during Recession
        # ------------------------------

        yearly_rec = recession_data.groupby(
            'Year'
        )['Automobile_Sales'].mean().reset_index()

        R_chart1 = dcc.Graph(
            figure=px.line(
                yearly_rec,
                x='Year',
                y='Automobile_Sales',
                title='Average Automobile Sales during Recession'
            )
        )

        # ------------------------------
        # Plot 2
        # Average Vehicles Sold by Type
        # ------------------------------

        average_sales = recession_data.groupby(
            'Vehicle_Type'
        )['Automobile_Sales'].mean().reset_index()

        R_chart2 = dcc.Graph(
            figure=px.bar(
                average_sales,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Vehicles Sold by Vehicle Type'
            )
        )

        # ------------------------------
        # Plot 3
        # Advertising Expenditure Share
        # ------------------------------

        exp_rec = recession_data.groupby(
            'Vehicle_Type'
        )['Advertising_Expenditure'].sum().reset_index()

        R_chart3 = dcc.Graph(
            figure=px.pie(
                exp_rec,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Advertisement Expenditure Share'
            )
        )

        # ------------------------------
        # Plot 4
        # Effect of Unemployment
        # ------------------------------

        unemp_data = recession_data.groupby(
            ['unemployment_rate', 'Vehicle_Type']
        )['Automobile_Sales'].mean().reset_index()

        R_chart4 = dcc.Graph(

            figure=px.bar(
                unemp_data,
                x='unemployment_rate',
                y='Automobile_Sales',
                color='Vehicle_Type',

                labels={
                    'unemployment_rate': 'Unemployment Rate',
                    'Automobile_Sales': 'Average Automobile Sales'
                },

                title='Effect of Unemployment Rate on Vehicle Type and Sales'
            )
        )

        return [

            html.Div(

                className='chart-item',

                children=[
                    html.Div(children=R_chart1),
                    html.Div(children=R_chart2)
                ],

                style={'display': 'flex'}
            ),

            html.Div(

                className='chart-item',

                children=[
                    html.Div(children=R_chart3),
                    html.Div(children=R_chart4)
                ],

                style={'display': 'flex'}
            )
        ]

    # ------------------------------------------------
    # Yearly Statistics Report
    # ------------------------------------------------

    elif (input_year and selected_statistics == 'Yearly Statistics'):

        yearly_data = data[data['Year'] == input_year]

        # ------------------------------
        # Plot 1
        # Yearly Automobile Sales
        # ------------------------------

        yas = data.groupby(
            'Year'
        )['Automobile_Sales'].mean().reset_index()

        Y_chart1 = dcc.Graph(

            figure=px.line(
                yas,
                x='Year',
                y='Automobile_Sales',
                title='Yearly Automobile Sales'
            )
        )

        # ------------------------------
        # Plot 2
        # Monthly Automobile Sales
        # ------------------------------

        mas = yearly_data.groupby(
            'Month'
        )['Automobile_Sales'].sum().reset_index()

        Y_chart2 = dcc.Graph(

            figure=px.line(
                mas,
                x='Month',
                y='Automobile_Sales',
                title='Total Monthly Automobile Sales'
            )
        )

        # ------------------------------
        # Plot 3
        # Average Vehicles Sold
        # ------------------------------

        avr_vdata = yearly_data.groupby(
            'Vehicle_Type'
        )['Automobile_Sales'].mean().reset_index()

        Y_chart3 = dcc.Graph(

            figure=px.bar(
                avr_vdata,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Vehicles Sold by Vehicle Type in the year {}'.format(input_year)
            )
        )

        # ------------------------------
        # Plot 4
        # Advertisement Expenditure
        # ------------------------------

        exp_data = yearly_data.groupby(
            'Vehicle_Type'
        )['Advertising_Expenditure'].sum().reset_index()

        Y_chart4 = dcc.Graph(

            figure=px.pie(
                exp_data,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Advertisement Expenditure for Each Vehicle'
            )
        )

        return [

    html.Div([

        html.Div(
            children=Y_chart1,
            style={
                'width': '48%',
                'display': 'inline-block'
            }
        ),

        html.Div(
            children=Y_chart2,
            style={
                'width': '48%',
                'display': 'inline-block'
            }
        )

    ]),

    html.Div([

        html.Div(
            children=Y_chart3,
            style={
                'width': '48%',
                'display': 'inline-block'
            }
        ),

        html.Div(
            children=Y_chart4,
            style={
                'width': '48%',
                'display': 'inline-block'
            }
        )

    ])

]

# Run the Dashboard Application

Run this cell and open the localhost link generated in the output.

In [15]:
# Run the app

if __name__ == '__main__':
    app.run()